Import necessary libraries

In [197]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Read in the csv-file

In [198]:
weather_data = pd.read_csv("weather_burbank_airport.csv")

## Data Dimensionality

In [199]:
weather_data.info(show_counts=True)
weather_data.head(24)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29244 entries, 0 to 29243
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   city                     29244 non-null  object 
 1   timestamp                29244 non-null  object 
 2   temperature              29219 non-null  float64
 3   cloud_cover              29224 non-null  float64
 4   cloud_cover_description  29224 non-null  object 
 5   pressure                 29236 non-null  float64
 6   windspeed                29158 non-null  float64
 7   precipitation            29244 non-null  float64
 8   felt_temperature         29218 non-null  float64
dtypes: float64(6), object(3)
memory usage: 2.0+ MB


,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 08:53:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 09:53:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 10:53:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 11:53:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 12:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0
5,Burbank,2018-01-01 13:53:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0
6,Burbank,2018-01-01 14:53:00,7.0,30.0,Partly Cloudy,992.08,0.0,0.0,7.0
7,Burbank,2018-01-01 15:53:00,8.0,34.0,Fair,992.41,0.0,0.0,8.0
8,Burbank,2018-01-01 16:53:00,12.0,34.0,Fair,993.39,0.0,0.0,12.0
9,Burbank,2018-01-01 17:53:00,16.0,34.0,Fair,994.05,0.0,0.0,16.0


## Data Conversions

Every timestamp is timezone-naive and needs to be converted to local time.

In [200]:
date_columns = ["timestamp"]
tz = "America/Los_Angeles"

for data in date_columns:
    weather_data[data] = pd.to_datetime(weather_data[data])

def convert_to_local(row, n=0): 
    for col in date_columns:
        ts = row[col]
        if pd.isna(ts):
            continue 
        row[col] = ts.tz_localize("UTC").tz_convert(tz) # Use tz_localize since timestamps are timezone-naive
    return row 

weather_data = weather_data.apply(convert_to_local, axis=1)

weather_data.info()
weather_data.head(24)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29244 entries, 0 to 29243
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   city                     29244 non-null  object                             
 1   timestamp                29244 non-null  datetime64[ns, America/Los_Angeles]
 2   temperature              29219 non-null  float64                            
 3   cloud_cover              29224 non-null  float64                            
 4   cloud_cover_description  29224 non-null  object                             
 5   pressure                 29236 non-null  float64                            
 6   windspeed                29158 non-null  float64                            
 7   precipitation            29244 non-null  float64                            
 8   felt_temperature         29218 non-null  float64                   

,city,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
0,Burbank,2018-01-01 00:53:00-08:00,9.0,33.0,Fair,991.75,9.0,0.0,8.0
1,Burbank,2018-01-01 01:53:00-08:00,9.0,33.0,Fair,992.08,0.0,0.0,9.0
2,Burbank,2018-01-01 02:53:00-08:00,9.0,21.0,Haze,992.08,0.0,0.0,9.0
3,Burbank,2018-01-01 03:53:00-08:00,9.0,29.0,Partly Cloudy,992.08,0.0,0.0,9.0
4,Burbank,2018-01-01 04:53:00-08:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0
5,Burbank,2018-01-01 05:53:00-08:00,8.0,33.0,Fair,992.08,0.0,0.0,8.0
6,Burbank,2018-01-01 06:53:00-08:00,7.0,30.0,Partly Cloudy,992.08,0.0,0.0,7.0
7,Burbank,2018-01-01 07:53:00-08:00,8.0,34.0,Fair,992.41,0.0,0.0,8.0
8,Burbank,2018-01-01 08:53:00-08:00,12.0,34.0,Fair,993.39,0.0,0.0,12.0
9,Burbank,2018-01-01 09:53:00-08:00,16.0,34.0,Fair,994.05,0.0,0.0,16.0


Drop 'city' column, since it does not provide any value

In [201]:
weather_data.drop(columns='city', inplace=True)

## Data Inconsistencies

Check for missing values

In [202]:
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29244 entries, 0 to 29243
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   timestamp                29244 non-null  datetime64[ns, America/Los_Angeles]
 1   temperature              29219 non-null  float64                            
 2   cloud_cover              29224 non-null  float64                            
 3   cloud_cover_description  29224 non-null  object                             
 4   pressure                 29236 non-null  float64                            
 5   windspeed                29158 non-null  float64                            
 6   precipitation            29244 non-null  float64                            
 7   felt_temperature         29218 non-null  float64                            
dtypes: datetime64[ns, America/Los_Angeles](1), float64(6), object(1)
me

Result: Missing values in columns several columns

### Timestamps

Check for possible time gaps.
Inconsistent timestamps are a strong indicator for missing data rows.

In [203]:
times_gaps = weather_data["timestamp"] - weather_data["timestamp"].shift(1)
missingTimes = times_gaps[times_gaps > times_gaps.iloc[1]]
missingTimes

2682    0 days 03:00:00
3402    0 days 04:00:00
3425    0 days 03:00:00
4324    0 days 02:00:00
5839    0 days 06:00:00
6206    0 days 06:00:00
7615    0 days 02:00:00
8896    0 days 02:00:00
12888   0 days 02:00:00
13919   0 days 17:00:00
15501   0 days 02:00:00
16839   0 days 02:00:00
17568   0 days 02:00:00
19688   0 days 02:00:00
20745   0 days 02:00:00
23637   0 days 01:15:00
26148   0 days 01:03:00
26656   0 days 02:00:00
27887   1 days 14:00:00
27915   0 days 02:00:00
28096   0 days 07:30:00
28597   0 days 03:00:00
Name: timestamp, dtype: timedelta64[ns]

Example for time gap with missing timestamps and data rows:

In [204]:
weather_data.loc[27886:27887]

,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
27886,2020-11-07 17:53:00-08:00,13.0,33.0,Fair,978.91,9.0,0.0,13.0
27887,2020-11-09 07:53:00-08:00,8.0,34.0,Fair,989.44,0.0,0.0,8.0


The time gaps show gaps that are not at the full hour level. Hence, there must be other data points in addition to the regular data points collected at minute 53 of every hour. These additional data points will be removed from the data set.

In [205]:
def removeMinorValues(tdf):
    for index, row in tdf.iterrows():
        if(row["timestamp"].minute != 53):
            tdf.drop(index, inplace=True)
    return tdf

weather_data = removeMinorValues(weather_data)
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26209 entries, 0 to 29243
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   timestamp                26209 non-null  datetime64[ns, America/Los_Angeles]
 1   temperature              26194 non-null  float64                            
 2   cloud_cover              26191 non-null  float64                            
 3   cloud_cover_description  26191 non-null  object                             
 4   pressure                 26201 non-null  float64                            
 5   windspeed                26129 non-null  float64                            
 6   precipitation            26209 non-null  float64                            
 7   felt_temperature         26193 non-null  float64                            
dtypes: datetime64[ns, America/Los_Angeles](1), float64(6), object(1)
memory 

Add rows and timestamps for regular data points

In [206]:
def addMissingRows(tdf):
    tdf = tdf.sort_values(by="timestamp").reset_index(drop=True)
    rows = []
    for i in range(len(tdf)-1):
        rows.append(tdf.iloc[i])
        currentTime = tdf.iloc[i]["timestamp"]
        nextTime = tdf.iloc[i+1]["timestamp"]
        while(nextTime-currentTime > pd.Timedelta(hours=1)):
            currentTime+= pd.Timedelta(hours=1)
            rows.append(pd.Series({"timestamp": currentTime, **{col: pd.NA for col in tdf.columns if col != "timestamp"}}))
    rows.append(tdf.iloc[len(tdf)-1])
    newDf = pd.DataFrame(rows)
    newDf = newDf.sort_values(by="timestamp").reset_index(drop=True)
    return newDf

weather_data = addMissingRows(weather_data)
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   timestamp                26304 non-null  datetime64[ns, America/Los_Angeles]
 1   temperature              26194 non-null  object                             
 2   cloud_cover              26191 non-null  object                             
 3   cloud_cover_description  26191 non-null  object                             
 4   pressure                 26201 non-null  object                             
 5   windspeed                26129 non-null  object                             
 6   precipitation            26209 non-null  object                             
 7   felt_temperature         26193 non-null  object                             
dtypes: datetime64[ns, America/Los_Angeles](1), object(7)
memory usage: 

### Interpolation

**Linear Interpolation**

Linear interpolation of float values that are not highly sensitive to the time of day. This includes:
- Pressure
- Cloud cover
- Windspeed
- Precipitation

In [207]:
def linear_interpolation(column):
    out = weather_data.copy()
    out[column] = pd.to_numeric(out[column], errors="coerce")
    out[column] = out[column].interpolate(method="linear", limit_area="inside")
    return out[column]

col_linear_interpolation = ["pressure", "cloud_cover", "precipitation", "windspeed"]
for col in col_linear_interpolation:
    weather_data[col] = linear_interpolation(col)
    weather_data[col] = weather_data[col].round(0)

# weather_data.info()
# weather_data[2394:2398]

**Time-Dependent Interpolation**

Temperature is highly dependent on the time of day. Therefore, we decided not to use simple linear interpolation, but use interpolate using the last and next available value from the specific hour.

This technique has advantages especially for the longer gaps since linear interpolation could miss local minima/maxima. For very short gaps, linear interpolation may be slightly more accurate, but in totality we find this time-of-day dependent interpolation more applicable.

A combination of both techniques is possible but will not be applied here due to minimal expected accuracy gains and a high expected cost (labour & complexity) per accuracy gain.

In [208]:
def time_dependent_interpolation(column):
    out = weather_data.copy()
    out[column] = pd.to_numeric(out[column], errors="coerce")
    out = out.sort_values("timestamp").set_index("timestamp")
    out[column] = (out[column].groupby(out.index.hour).transform(lambda s: s.interpolate(method="time", limit_area="inside")))
    out = out.reset_index(drop=True)
    return out[column]


col_time_dependent = ["temperature", "felt_temperature"]

for col in col_time_dependent:
    weather_data[col] = time_dependent_interpolation(col)
    weather_data[col] = weather_data[col].round(0)

weather_data.info()
weather_data[25001:25043]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   timestamp                26304 non-null  datetime64[ns, America/Los_Angeles]
 1   temperature              26304 non-null  float64                            
 2   cloud_cover              26304 non-null  float64                            
 3   cloud_cover_description  26191 non-null  object                             
 4   pressure                 26304 non-null  float64                            
 5   windspeed                26304 non-null  float64                            
 6   precipitation            26304 non-null  float64                            
 7   felt_temperature         26304 non-null  float64                            
dtypes: datetime64[ns, America/Los_Angeles](1), float64(6), object(1)
me

,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
25001,2020-11-07 17:53:00-08:00,13.0,33.0,Fair,979.0,9.0,0.0,13.0
25002,2020-11-07 18:53:00-08:00,15.0,33.0,<NA>,979.0,9.0,0.0,15.0
25003,2020-11-07 19:53:00-08:00,15.0,33.0,<NA>,979.0,9.0,0.0,15.0
25004,2020-11-07 20:53:00-08:00,14.0,33.0,<NA>,980.0,8.0,0.0,14.0
25005,2020-11-07 21:53:00-08:00,13.0,33.0,<NA>,980.0,8.0,0.0,13.0
25006,2020-11-07 22:53:00-08:00,12.0,33.0,<NA>,980.0,8.0,0.0,12.0
25007,2020-11-07 23:53:00-08:00,12.0,33.0,<NA>,981.0,8.0,0.0,11.0
25008,2020-11-08 00:53:00-08:00,11.0,33.0,<NA>,981.0,7.0,0.0,11.0
25009,2020-11-08 01:53:00-08:00,11.0,33.0,<NA>,981.0,7.0,0.0,11.0
25010,2020-11-08 02:53:00-08:00,10.0,33.0,<NA>,981.0,7.0,0.0,10.0


**Cloud Cover Description - Mapping**

We already interpolated the cloud cover values, but the cloud cover description values still contains missing values.

Therefore, we use a mapping of cloud cover descriptions based on the cloud cover values.

In [209]:
wd_copy = weather_data.copy()
cc_mapping = (
    wd_copy.groupby("cloud_cover")["cloud_cover_description"]
      .apply(lambda s: sorted(s.dropna().unique().tolist()))
      .to_dict()
)
cc_mapping

{4.0: ['Heavy T-Storm', 'T-Storm'],
 11.0: ['Light Rain', 'Light Rain / Windy'],
 12.0: ['Rain'],
 19.0: ['Blowing Dust'],
 20.0: ['Fog'],
 21.0: ['Haze'],
 22.0: ['Smoke'],
 23.0: [],
 25.0: [],
 26.0: ['Cloudy', 'Cloudy / Windy'],
 27.0: ['Mostly Cloudy', 'Mostly Cloudy / Windy'],
 28.0: ['Mostly Cloudy', 'Mostly Cloudy / Windy'],
 29.0: ['Partly Cloudy', 'Partly Cloudy / Windy'],
 30.0: ['Partly Cloudy', 'Partly Cloudy / Windy'],
 31.0: [],
 32.0: [],
 33.0: ['Fair', 'Fair / Windy'],
 34.0: ['Fair', 'Fair / Windy'],
 38.0: ['Thunder in the Vicinity'],
 40.0: ['Heavy Rain', 'Heavy Rain / Windy'],
 47.0: ['Thunder in the Vicinity']}

Due to the linear interpolation, we can observe some cloud cover values that do not have a mapping to a description.

Since it is only a very limited number of cases, we will interpolate with the closest cloud cover description and adapt the mapping manually.

In [210]:
cc_mapping[23.0] = ["Smoke"]
cc_mapping[25.0] = ["Cloudy"]
cc_mapping[31.0] = ["Partly Cloudy"]
cc_mapping[32.0] = ["Fair"]

In case there are multiple descriptions for a cloud cover, pick the first occurence

In [211]:
cc_mapping = {
    k: v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else v
    for k, v in cc_mapping.items()
}

Apply the mapping

In [212]:
weather_data["cloud_cover_description"] = weather_data["cloud_cover_description"].fillna(weather_data["cloud_cover"].map(cc_mapping))
weather_data.info()
weather_data[2394:2398]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype                              
---  ------                   --------------  -----                              
 0   timestamp                26304 non-null  datetime64[ns, America/Los_Angeles]
 1   temperature              26304 non-null  float64                            
 2   cloud_cover              26304 non-null  float64                            
 3   cloud_cover_description  26304 non-null  object                             
 4   pressure                 26304 non-null  float64                            
 5   windspeed                26304 non-null  float64                            
 6   precipitation            26304 non-null  float64                            
 7   felt_temperature         26304 non-null  float64                            
dtypes: datetime64[ns, America/Los_Angeles](1), float64(6), object(1)
me

,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
2394,2018-04-10 19:53:00-07:00,24.0,29.0,Partly Cloudy,986.0,11.0,0.0,24.0
2395,2018-04-10 20:53:00-07:00,23.0,29.0,Partly Cloudy,987.0,10.0,0.0,22.0
2396,2018-04-10 21:53:00-07:00,21.0,29.0,Partly Cloudy,987.0,8.0,0.0,21.0
2397,2018-04-10 22:53:00-07:00,22.0,29.0,Partly Cloudy,987.0,7.0,0.0,22.0


### Outliers

There seem to be no apparent outlier values within the dataset.

In [213]:
weather_data.describe()

,temperature,cloud_cover,pressure,windspeed,precipitation,felt_temperature
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000
mean,18.257565,30.788587,986.860782,8.474110,0.040146,18.095993
std,6.569617,4.689051,3.586114,6.708769,0.408796,6.423847
min,2.000000,4.000000,971.000000,0.000000,0.000000,0.000000
25%,13.000000,28.000000,984.000000,6.000000,0.000000,13.000000
50%,18.000000,33.000000,986.000000,7.000000,0.000000,18.000000
75%,22.000000,34.000000,989.000000,13.000000,0.000000,22.000000
max,46.000000,47.000000,1000.000000,57.000000,19.000000,42.000000


## File Export

In [214]:
weather_data.to_csv("../00_Productive_Data/weather_data_clean.csv")